Question 27. Perform left, right and full outer joins on Q26's data1set and data2set. Compare each with the inner join.

In [1]:
# Step 1: Import the libraries needed for this program.
import pandas as pd
from IPython.display import display

# Step 2: Load the CSV file from this question's folder into a DataFrame (table).
df = pd.read_csv("employee_SalesData.csv")
print("Original dataset:")
display(df)

# Step 3: Recreate Q26's first fragment without employee 1004.
data1set = df.loc[df["Emp_ID"] != 1004, ["Emp_ID", "Name", "Department", "Year"]].copy()

# Step 4: Recreate Q26's second fragment without employee 1002.
data2set = df.loc[df["Emp_ID"] != 1002, ["Emp_ID", "Region", "Product", "Sales"]].copy()

# Step 5: Save both input fragments.
data1set.to_csv("data1set.csv", index=False)
data2set.to_csv("data2set.csv", index=False)


# Step 6: Create the inner join as a reference for comparison.
inner = data1set.merge(data2set, on="Emp_ID", how="inner", validate="one_to_one")

# Step 7: Run all four join types; _merge explains which side supplied each record.
summary = []
for join_type in ["inner", "left", "right", "outer"]:
    joined = data1set.merge(
        data2set, on="Emp_ID", how=join_type, indicator=True, validate="one_to_one"
    )
    print(f"\n{join_type.upper()} JOIN")
    display(joined)
    joined.to_csv(f"{join_type}_join.csv", index=False)
    summary.append({
        "Join": join_type, "Rows": len(joined),
        "IDs": ", ".join(joined["Emp_ID"].astype(str)),
        "Added_vs_inner": ", ".join(map(str, sorted(set(joined["Emp_ID"]) - set(inner["Emp_ID"])))),
    })

# Step 8: Compare row counts and employee IDs added beyond the inner join.
result = pd.DataFrame(summary)
display(result)

# Final step: Save the answer without the automatic row-number column.
result.to_csv("result.csv", index=False)
print("Saved result.csv")


Original dataset:


,Emp_ID,Name,Department,Region,Product,Sales,Year
0,1001,Alisha,Finance,North,Laptop,50000,2025
1,1002,Ravi,IT,South,Phone,20000,2026
2,1003,Neha,Finance,East,Tablet,15000,2024
3,1004,Salim,IT,West,Laptop,52000,2025
4,1005,Priya,IT,North,Phone,21000,2025
5,1006,Alvin,Finance,South,Tablet,18000,2024



INNER JOIN

,Emp_ID,Name,Department,Year,Region,Product,Sales,_merge
0,1001,Alisha,Finance,2025,North,Laptop,50000,both
1,1003,Neha,Finance,2024,East,Tablet,15000,both
2,1005,Priya,IT,2025,North,Phone,21000,both
3,1006,Alvin,Finance,2024,South,Tablet,18000,both



LEFT JOIN


,Emp_ID,Name,Department,Year,Region,Product,Sales,_merge
0,1001,Alisha,Finance,2025,North,Laptop,50000.0,both
1,1002,Ravi,IT,2026,NaN,NaN,NaN,left_only
2,1003,Neha,Finance,2024,East,Tablet,15000.0,both
3,1005,Priya,IT,2025,North,Phone,21000.0,both
4,1006,Alvin,Finance,2024,South,Tablet,18000.0,both


... (2 outputs left out)


OUTER JOIN


,Emp_ID,Name,Department,Year,Region,Product,Sales,_merge
0,1001,Alisha,Finance,2025.0,North,Laptop,50000.0,both
1,1002,Ravi,IT,2026.0,NaN,NaN,NaN,left_only
2,1003,Neha,Finance,2024.0,East,Tablet,15000.0,both
3,1004,NaN,NaN,NaN,West,Laptop,52000.0,right_only
4,1005,Priya,IT,2025.0,North,Phone,21000.0,both
5,1006,Alvin,Finance,2024.0,South,Tablet,18000.0,both


,Join,Rows,IDs,Added_vs_inner
0,inner,4,"1001, 1003, 1005, 1006",
1,left,5,"1001, 1002, 1003, 1005, 1006",1002
2,right,5,"1001, 1003, 1004, 1005, 1006",1004
3,outer,6,"1001, 1002, 1003, 1004, 1005, 1006","1002, 1004"


Saved result.csv


Interpretation / method

Inner: 4 rows. Left: 5, including 1002 with missing right-side values. Right: 5, including 1004 with missing left-side values. Full outer: all 6 IDs; _merge identifies unmatched rows.